In [6]:
import os
import sys
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

sys.path.insert(0, "../src")  # make validation/src importable
from build_amy_epw import AMYEPWBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

load_dotenv("../.env")
NREL_API_KEY = os.environ["NREL_API_KEY"]
NREL_EMAIL   = os.environ["NREL_EMAIL"]

YEAR    = 2023
OUT_DIR = Path("../src/data/weather")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Build valid site list from HEMS-RBSA

In [7]:
all_info = pd.read_csv(r"../src/data/HEMS_RBSA/HEMS_RBSA_BASE.csv")
key_columns = [
    "Conditioned_Area", "Qty_Occupants", "Primary_Heating_System_Type",
    "Total_Wall_U-Value", "Window_U-Value", "Window_to_Wall_Ratio",
]
invalid = {"Not Available", "Unknown", "unknown", "not available", "N/A", "n/a", ""}

def is_valid(val):
    if pd.isna(val):
        return False
    return str(val).strip() not in invalid

mask = all_info[key_columns].apply(lambda col: col.map(is_valid)).all(axis=1)
all_info_clean = all_info[mask].copy()
ee_id_rbsa_clean = all_info_clean["ee_site_id"].drop_duplicates().reset_index(drop=True)

print(f"Kept {len(all_info_clean)} / {len(all_info)} rows")
print(f"{len(ee_id_rbsa_clean)} valid sites")

Kept 293 / 377 rows
293 valid sites


## Get unique station_ids for valid sites

In [8]:
sites_meta = pd.read_csv(r"../src/data/metadata/SITES v9.2.csv", dtype={"station_id": str})

def normalize_station_id(station_id):
    station_id = str(station_id).strip()
    if "-" not in station_id:
        return station_id
    usaf, wban = station_id.split("-", 1)
    return f"{usaf}-{wban.zfill(5)}"

valid_station_ids_raw = (
    sites_meta[sites_meta["ee_site_id"].isin(ee_id_rbsa_clean)]
    ["station_id"]
    .dropna()
    .unique()
)
valid_station_ids = pd.Index(valid_station_ids_raw).map(normalize_station_id).unique()
station_id_normalization = (
    pd.DataFrame({"station_id_raw": valid_station_ids_raw})
    .assign(station_id=lambda df: df["station_id_raw"].map(normalize_station_id))
)

missing = set(ee_id_rbsa_clean) - set(sites_meta["ee_site_id"])
if missing:
    print(f"WARNING: {len(missing)} site(s) have no station mapping: {missing}")

print(f"{len(valid_station_ids)} unique station IDs for {len(ee_id_rbsa_clean)} valid sites")
print(valid_station_ids)

60 unique station IDs for 293 valid sites
Index(['999999-04136', '727856-94176', '727850-24157', '742060-24207',
       '720638-00224', '720202-00118', '726813-94195', '725895-94236',
       '726930-24221', '727937-24222', '727870-94119', '727846-24160',
       '727918-94298', '726986-94261', '720388-00469', '727834-24136',
       '720322-04129', '720923-00310', '726881-94273', '726985-24242',
       '727935-24234', '726980-24229', '994350-99999', '720734-00264',
       '726920-24230', '727945-04205', '727845-24163', '726836-04201',
       '727810-24243', '726810-24131', '727938-94274', '725970-24225',
       '725866-94178', '720749-24255', '727884-99999', '727924-24223',
       '727730-24153', '726945-24202', '727930-24233', '727934-94248',
       '727855-24114', '727925-94227', '727976-24217', '727857-94129',
       '727790-24146', '742071-24201', '726959-94281', '999999-04141',
       '726904-24231', '727873-99999', '725785-24145', '726950-24285',
       '727883-24220', '727830-2414

## Build EPW files — one per station_id

In [9]:
errors = {}

for station_id in tqdm(valid_station_ids, desc="Building EPW files"):
    out_path = OUT_DIR / f"{station_id}.epw"
    legacy_station_ids = station_id_normalization.loc[
        station_id_normalization["station_id"] == station_id, "station_id_raw"
    ].tolist()
    legacy_out_paths = [OUT_DIR / f"{legacy_station_id}.epw" for legacy_station_id in legacy_station_ids]

    if out_path.exists():
        print(f"  SKIP  {station_id}  (already exists)")
        continue

    existing_legacy_path = next((path for path in legacy_out_paths if path.exists()), None)
    if existing_legacy_path is not None:
        existing_legacy_path.rename(out_path)
        print(f"  RENAME  {existing_legacy_path.name} -> {out_path.name}")
        continue

    try:
        builder = AMYEPWBuilder(station_id, YEAR, NREL_API_KEY, NREL_EMAIL)
        builder.build(str(out_path))
        print(f"  OK    {station_id} -> {out_path}")
    except Exception as exc:
        errors[station_id] = exc
        print(f"  FAIL  {station_id}: {exc}")

print(f"Done -- {len(valid_station_ids) - len(errors)} succeeded, {len(errors)} failed")
if errors:
    print("Failed station IDs:", list(errors.keys()))

Building EPW files:   0%|          | 0/60 [00:00<?, ?it/s]INFO  === AMYEPWBuilder: station=999999-04136, year=2023 ===
INFO  [1/4] Fetching station metadata
INFO  Station: SPOKANE 17 SSW  (47.4170, -117.5260)  elev=691 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 105473 raw observations downloaded
WARNING  ISD: GD1 cloud field not present — total_sky_cover will default to 5 (50%%) in EPW
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=47.4170, lon=-117.5260, year=2023
INFO  NSRDB: 8760 rows, TZ offset=-8
INFO  [4/4] Writing EPW → ..\src\data\weather\999999-04136.epw
INFO  EPW written: ..\src\data\weather\999999-04136.epw
INFO  Done → ..\src\data\weather\999999-04136.epw
Building EPW files:   2%|▏         | 1/60 [00:12<12:31, 12.73s/it]INFO  === AMYEPWBuilder: station=720638-00224, year=2023 ===
INFO  [1/4] Fetching station metadata


  OK    999999-04136 -> ..\src\data\weather\999999-04136.epw
  SKIP  727856-94176  (already exists)
  SKIP  727850-24157  (already exists)
  SKIP  742060-24207  (already exists)


INFO  Station: BEND MUNICIPAL AIRPORT  (44.0950, -121.2000)  elev=1055 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 25893 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=44.0950, lon=-121.2000, year=2023
INFO  NSRDB: 8760 rows, TZ offset=-8
INFO  [4/4] Writing EPW → ..\src\data\weather\720638-00224.epw
INFO  EPW written: ..\src\data\weather\720638-00224.epw
INFO  Done → ..\src\data\weather\720638-00224.epw
Building EPW files:   8%|▊         | 5/60 [00:21<03:28,  3.78s/it]INFO  === AMYEPWBuilder: station=720202-00118, year=2023 ===
INFO  [1/4] Fetching station metadata


  OK    720638-00224 -> ..\src\data\weather\720638-00224.epw


INFO  Station: TILLAMOOK AIRPORT  (45.4170, -123.8170)  elev=11 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 25687 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=45.4170, lon=-123.8170, year=2023
INFO  NSRDB: 8760 rows, TZ offset=-8
INFO  [4/4] Writing EPW → ..\src\data\weather\720202-00118.epw
INFO  EPW written: ..\src\data\weather\720202-00118.epw
INFO  Done → ..\src\data\weather\720202-00118.epw
Building EPW files:  10%|█         | 6/60 [00:27<03:56,  4.38s/it]INFO  === AMYEPWBuilder: station=720388-00469, year=2023 ===
INFO  [1/4] Fetching station metadata


...

  OK    726836-04201 -> ..\src\data\weather\726836-04201.epw
  SKIP  727810-24243  (already exists)
  SKIP  726810-24131  (already exists)
  SKIP  727938-94274  (already exists)
  SKIP  725970-24225  (already exists)
  SKIP  725866-94178  (already exists)
  SKIP  720749-24255  (already exists)
  SKIP  727884-99999  (already exists)
  SKIP  727924-24223  (already exists)
  SKIP  727730-24153  (already exists)
  SKIP  726945-24202  (already exists)
  SKIP  727930-24233  (already exists)
  SKIP  727934-94248  (already exists)
  SKIP  727855-24114  (already exists)
  SKIP  727925-94227  (already exists)
  SKIP  727976-24217  (already exists)
  SKIP  727857-94129  (already exists)
  SKIP  727790-24146  (already exists)
  SKIP  742071-24201  (already exists)
  SKIP  726959-94281  (already exists)


INFO  Station: COOS BAY 8 SW  (43.2720, -124.3190)  elev=4 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 104573 raw observations downloaded
WARNING  ISD: GD1 cloud field not present — total_sky_cover will default to 5 (50%%) in EPW
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=43.2720, lon=-124.3190, year=2023
INFO  NSRDB: 8760 rows, TZ offset=-8
INFO  [4/4] Writing EPW → ..\src\data\weather\999999-04141.epw
INFO  EPW written: ..\src\data\weather\999999-04141.epw
INFO  Done → ..\src\data\weather\999999-04141.epw
Building EPW files:  80%|████████  | 48/60 [01:15<00:10,  1.14it/s]INFO  === AMYEPWBuilder: station=726883-04113, year=2023 ===
INFO  [1/4] Fetching station metadata


  OK    999999-04141 -> ..\src\data\weather\999999-04141.epw
  SKIP  726904-24231  (already exists)
  SKIP  727873-99999  (already exists)
  SKIP  725785-24145  (already exists)
  SKIP  726950-24285  (already exists)
  SKIP  727883-24220  (already exists)
  SKIP  727830-24149  (already exists)


INFO  Station: HERMISTON MUNICIPAL ARPT  (45.8260, -119.2610)  elev=195 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 10221 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=45.8260, lon=-119.2610, year=2023
INFO  NSRDB: 8760 rows, TZ offset=-8
INFO  [4/4] Writing EPW → ..\src\data\weather\726883-04113.epw
INFO  EPW written: ..\src\data\weather\726883-04113.epw
INFO  Done → ..\src\data\weather\726883-04113.epw
Building EPW files: 100%|██████████| 60/60 [01:21<00:00,  1.36s/it]

  OK    726883-04113 -> ..\src\data\weather\726883-04113.epw
  SKIP  725975-24235  (already exists)
  SKIP  994025-99999  (already exists)
  SKIP  726940-24232  (already exists)
  SKIP  726917-24284  (already exists)
  SKIP  727928-94263  (already exists)
Done -- 60 succeeded, 0 failed
